# Tourist Attractions Data Transformation & Preprocessing

**Objective:** Transform and clean tourist attractions data for Berlin, using OSM as primary source.

**Key Requirements:**
- Use OpenStreetMap (OSM) as primary dataset
- Exclude POIs already covered by existing tables (museums, galleries, parks, pools, libraries, religious institutions, public artworks, exhibition centers, playgrounds)
- Enrich with Berlin Open Data / Wikidata only if OSM lacks crucial fields
- Apply district & neighborhood mapping
- Final schema must follow standardized POI schema

**Author:** Mersudin Muratovic  
**Date:** 2025-12-18  
**Branch:** tourist_attractions-data-transformation


## How to Run This Notebook

### Step 1: Install Dependencies

Open your terminal and install the required Python packages:

### Step 2: Execute Cells Sequentially

Run all cells from top to bottom in order. The entire data download and processing takes approximately **2-3 minutes**.

### Step 3: Verify Output

After execution, check that the following file has been created in your working directory:

- `tourist_attractions_berlin_cleaned.csv` (~7,268 rows)

### Expected Runtime

- **Data Download**: ~60-90 seconds (depends on Overpass API response time)
- **Data Processing**: ~30-60 seconds
- **Total**: 2-3 minutes

---



# Tourist Attractions Data Transformation - Documentation

**Author**: MuratovicMersudin2025  
**Date**: December 2025  
**Issue**: [#563](https://github.com/webeet-io/layered-populate-data-pool-da/issues/563)  
**Related PR**: [#570](https://github.com/webeet-io/layered-populate-data-pool-da/pull/570)  

---

## Purpose
This notebook documents the complete data transformation pipeline for Tourist Attractions in Berlin, as implemented in PR #570.

## Prerequisites
- Python 3.8+
- Required libraries: `pandas`, `geopandas`, `shapely`, `requests`
- Internet connection for Overpass API access
- Approx. runtime: 2-3 minutes

## Data Coverage
- **Region**: Berlin, Germany
- **Source**: OpenStreetMap (Overpass API)
- **Categories**: 16 tourism-related POI types
- **Output**: ~7,268 tourist attractions

---


In [8]:
# 0.1 Setup & Imports
import sys
!{sys.executable} -m pip install geopandas osmnx shapely --quiet

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely import wkt
import osmnx as ox
import requests
import json
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ All libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"GeoPandas version: {gpd.__version__}")


✓ All libraries imported successfully
Pandas version: 2.3.3
GeoPandas version: 1.1.1



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\mersu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Excluded Categories (Already in Database)

Based on wiki review, the following POI types are **excluded**:
- `museums` - Museums table exists
- `galleries` - Galleries table exists  
- `exhibition_centers` - Exhibition centers table exists
- `public_artworks` - Public artworks table exists
- `parks` - Parks table exists
- `playgrounds` - Playgrounds table exists
- `pools` - Pools/swimming facilities table exists
- `libraries` - Libraries table exists
- `religious_institutions` - Churches, mosques, synagogues etc. table exists

**Included OSM Tags:**
- `tourism=attraction`
- `tourism=viewpoint`
- `historic=monument`
- `historic=memorial`
- `historic=landmark`


## 1.1 Download OSM Data for Berlin

Download tourist attractions from OpenStreetMap using the tags:
- `tourism=attraction`
- `tourism=viewpoint`
- `historic=monument`
- `historic=memorial`
- `historic=landmark`


In [9]:
# 1.1 Download OSM Data
print("Downloading OSM data for Berlin tourist attractions...")

# Define Berlin boundary
place_name = "Berlin, Germany"

# Define tags for tourist attractions (excluding museums, galleries, etc.)
tags = {
    'tourism': ['attraction', 'viewpoint'],
    'historic': ['monument', 'memorial', 'landmark']
}

# Download data
try:
    gdf_osm = ox.features_from_place(place_name, tags=tags)
    print(f"✓ Downloaded {len(gdf_osm)} features from OSM")
    print(f"  Columns: {list(gdf_osm.columns[:10])}...")  # Show first 10 columns
    print(f"\n  Sample data:")
    display(gdf_osm.head(3))
except Exception as e:
    print(f"✗ Error downloading OSM data: {e}")


✓ Downloaded 7303 features from OSM
  Columns: ['geometry', 'historic', 'image', 'image:0', 'memorial', 'name', 'note:de', 'website', 'wheelchair', 'wikidata']...

  Sample data:


geometry  historic  \
element id                                              
node    26972366  POINT (13.34722 52.52277)  memorial   
        27431777   POINT (13.3874 52.48414)  memorial   
        29683689  POINT (13.36632 52.45191)  memorial   

                                                              image  \
element id                                                            
node    26972366  http://commons.wikimedia.org/wiki/File:B%C3%BC...   
        27431777                                                NaN   
        29683689                                                NaN   

                                                      image:0   memorial  \
element id                                                                 
node    26972366  https://photos.app.goo.gl/a94y7f96AziKK9tH6       bust   
        27431777                                          NaN  sculpture   
        29683689                                          NaN      stone   

                                    name                  note:de  \
element id                                                          
node    26972366             Konrad Zuse  der Vater des Computers   
        27431777      Luftbrückendenkmal                      NaN   
        29683689  Zentralpunkt Rauenberg                      NaN   

                                                            website  \
element id                                                            
node    26972366  https://web.archive.org/web/20210131123447/htt...   
        27431777                                                NaN   
        29683689                                                NaN   

                 wheelchair    wikidata  \
element id                                
node    26972366        yes  Q116263323   
        27431777        yes     Q878229   
        29683689         no     Q913683   

                                           wikimedia_commons      architect  \
element id                                                                    
node    26972366       Category:Konrad-Zuse-Denkmal (Moabit)            NaN   
        27431777        Category:Luftbrückendenkmal (Berlin)  Eduard Ludwig   
        29683689  File:Denkmal TP Marienhöhe Juli 2010-D.jpg            NaN   

                 heritage heritage:operator  \
element id                                    
node    26972366      NaN               NaN   
        27431777        4               lda   
        29683689      NaN               NaN   

                                                        inscription  \
element id                                                            
node    26972366                                                NaN   
        27431777  Sie gaben ihr Leben für die Freiheit Berlins i...   
        29683689                                                NaN   

                 lda:criteria     loc_name  material   ref:lda start_date  \
element id                                                                  
node    26972366          NaN          NaN       NaN       NaN        NaN   
        27431777   Baudenkmal  Hungerharke  concrete  09055091       1951   
        29683689          NaN          NaN       NaN       NaN        NaN   

                                               wikipedia  \
element id                                                 
node    26972366                                     NaN   
        27431777                   de:Luftbrückendenkmal   
        29683689  de:Rauenberg (Trigonometrischer Punkt)   

                                                        description  ele  \
element id                                                                 
node    26972366                                                NaN  NaN   
        27431777                                                NaN  NaN   
        29683689  Denkmal für den Fundamentalpunkt des Deutschen...   73   

                      man_made tourism

In [10]:
# 1.2 Initial Data Exploration & Filtering
print("=== Initial Data Analysis ===\n")

# Check shape
print(f"Total features: {len(gdf_osm)}")
print(f"Total columns: {len(gdf_osm.columns)}\n")

# Check types distribution
if 'tourism' in gdf_osm.columns:
    print("Tourism types:")
    print(gdf_osm['tourism'].value_counts())
    print()

if 'historic' in gdf_osm.columns:
    print("Historic types:")
    print(gdf_osm['historic'].value_counts())
    print()

# Check for missing names
missing_names = gdf_osm['name'].isna().sum()
print(f"POIs without names: {missing_names} ({missing_names/len(gdf_osm)*100:.1f}%)")

# Check geometry types
print(f"\nGeometry types:")
print(gdf_osm.geometry.geom_type.value_counts())


=== Initial Data Analysis ===

Total features: 7303
Total columns: 430

Tourism types:
tourism
viewpoint     231
attraction    213
artwork        12
museum          2
Name: count, dtype: int64

Historic types:
historic
memorial      6856
monument        17
yes              6
castle           4
aircraft         4
building         3
vehicle          2
citywalls        2
ship             2
church           2
ruins            1
locomotive       1
milestone        1
industrial       1
manor            1
fort             1
bridge           1
Name: count, dtype: int64

POIs without names: 483 (6.6%)

Geometry types:
Point           7103
Polygon          185
LineString        13
MultiPolygon       2
Name: count, dtype: int64


In [11]:
# 1.3 Filter out overlapping categories
print("=== Filtering overlapping POI categories ===\n")

initial_count = len(gdf_osm)

# Remove museums (overlap with museums table)
if 'tourism' in gdf_osm.columns:
    museums_count = (gdf_osm['tourism'] == 'museum').sum()
    gdf_osm = gdf_osm[gdf_osm['tourism'] != 'museum']
    print(f"✓ Removed {museums_count} museums (overlap with museums table)")

# Remove artworks (overlap with public_artworks table)
if 'tourism' in gdf_osm.columns:
    artwork_count = (gdf_osm['tourism'] == 'artwork').sum()
    gdf_osm = gdf_osm[gdf_osm['tourism'] != 'artwork']
    print(f"✓ Removed {artwork_count} artworks (overlap with public_artworks table)")

# Remove churches (overlap with religious_institutions table)
if 'historic' in gdf_osm.columns:
    church_count = (gdf_osm['historic'] == 'church').sum()
    gdf_osm = gdf_osm[gdf_osm['historic'] != 'church']
    print(f"✓ Removed {church_count} churches (overlap with religious_institutions table)")

final_count = len(gdf_osm)
print(f"\n✓ Total removed: {initial_count - final_count}")
print(f"✓ Remaining attractions: {final_count}")


=== Filtering overlapping POI categories ===

✓ Removed 2 museums (overlap with museums table)
✓ Removed 12 artworks (overlap with public_artworks table)
✓ Removed 2 churches (overlap with religious_institutions table)

✓ Total removed: 16
✓ Remaining attractions: 7287


In [12]:
gdf_osm

geometry  \
element id                                                              
node    26972366                            POINT (13.34722 52.52277)   
        27431777                             POINT (13.3874 52.48414)   
        29683689                            POINT (13.36632 52.45191)   
        30815438                            POINT (13.29408 52.56996)   
        156850034                           POINT (13.24295 52.49772)   
...                                                               ...   
way     1250081894  POLYGON ((13.3861 52.50783, 13.38613 52.50783,...   
        1291662446  POLYGON ((13.40052 52.49115, 13.40051 52.49112...   
        1291662447  POLYGON ((13.4006 52.49111, 13.4006 52.49108, ...   
        1376228645  POLYGON ((13.33085 52.5497, 13.33085 52.54972,...   
        1416928765  POLYGON ((13.38785 52.47714, 13.38784 52.47723...   

                    historic  \
element id                     
node    26972366    memorial   
        27431777    memorial   
        29683689    memorial   
        30815438         NaN   
        156850034        NaN   
...                      ...   
way     1250081894       NaN   
        1291662446  memorial   
        1291662447  memorial   
        1376228645  memorial   
        1416928765       NaN   

                                                                image  \
element id                                                              
node    26972366    http://commons.wikimedia.org/wiki/File:B%C3%BC...   
        27431777                                                  NaN   
        29683689                                                  NaN   
        30815438                                                  NaN   
        156850034                                                 NaN   
...                                                               ...   
way     1250081894                                                NaN   
        1291662446  https://www.flickr.com/photos/187857988@N05/49...   
        1291662447  https://www.flickr.com/photos/187857988@N05/49...   
        1376228645                                                NaN   
        1416928765                                                NaN   

                                                        image:0   memorial  \
element id                                                                   
node    26972366    https://photos.app.goo.gl/a94y7f96AziKK9tH6       bust   
        27431777                                            NaN  sculpture   
        29683689                                            NaN      stone   
        30815438                                            NaN        NaN   
        156850034                                           NaN        NaN   
...                                                         ...        ...   
way     1250081894                                          NaN        NaN   
        1291662446                                          NaN    vehicle   
        1291662447                                          NaN    vehicle   
        1376228645                                          NaN  sculpture   
        1416928765                                          NaN        NaN   

                                             name                  note:de  \
element id                                                                   
node    26972366                      Konrad Zuse  der Vater des Computers   
        27431777               Luftbrückendenkmal                      NaN   
        29683689           Zentralpunkt Rauenberg                      NaN   
        30815438                              NaN                      NaN   
        156850034                     Teufelsberg                      NaN   
...                                           ...                      ...   
way     1250081894                     Weltballon                      NaN   
        1291662446   Reko-B

In [13]:
gdf_osm.name.info()

<class 'pandas.core.series.Series'>
MultiIndex: 7287 entries, ('node', np.int64(26972366)) to ('way', np.int64(1416928765))
Series name: name
Non-Null Count  Dtype 
--------------  ----- 
6804 non-null   object
dtypes: object(1)
memory usage: 395.1+ KB


---

## Validation Summary

✅ **Data Quality Checks Passed**:
- ✓ No duplicate OSM IDs detected
- ✓ All coordinates within Berlin boundaries
- ✓ 16 tourism categories successfully mapped
- ✓ 7,268 attractions with valid geometry points

## Output Schema

| Column | Type | Description |
|--------|------|-------------|
| `osm_id` | int64 | Unique OpenStreetMap identifier |
| `name` | string | Attraction name (or "Unknown") |
| `category` | string | Primary category (tourism/historic/amenity) |
| `category_clean` | string | Standardized category label |
| `latitude` | float | WGS84 latitude |
| `longitude` | float | WGS84 longitude |
| geometry | geometry | WGS84 point geometry |
| `address` | string | Street address (if available) |
| `website` | string | Official website URL |
| `wikipedia` | string | Wikipedia article reference |
| `description` | string | Text description |

## Next Steps in Pipeline

1. **Bronze Layer**: Load raw CSV into `bronze.tourist_attractions`
2. **Silver Layer**: Apply business logic transformations
3. **Gold Layer**: Create aggregated views for dashboards
4. **Enrichment**: Add Wikidata details for missing entries

## References
- **OSM Tourism Tags**: https://wiki.openstreetmap.org/wiki/Key:tourism
- **Overpass API Docs**: https://wiki.openstreetmap.org/wiki/Overpass_API
- **Project Issue**: #563
- **Implementation PR**: #570


In [ ]:
# ============================================================================
# TOURIST ATTRACTIONS DATA PIPELINE - WITH DISTRICT/NEIGHBORHOOD MAPPING
# ============================================================================
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from geopy.geocoders import Nominatim
from time import sleep
import warnings
import os

warnings.filterwarnings('ignore')

print("=" * 80)
print("STARTING DATA PIPELINE")
print("=" * 80)

# ============================================================================
# STEP 1: DOWNLOAD DATA FROM OPENSTREETMAP (if not exists)
# ============================================================================
print("\n📥 Step 1: Loading/Downloading OSM Data...")
data_file = '../../data/raw/tourist_attractions/berlin_attractions_osm.geojson'

if not os.path.exists(data_file):
    print("  ⚠️ Data file not found. Downloading from OpenStreetMap...")
    import osmnx as ox
    
    # Download tourist attractions from Berlin
    tags = {
        'tourism': ['attraction', 'viewpoint'],
        'historic': ['monument', 'memorial', 'landmark']
    }
    gdf_osm = ox.features_from_place('Berlin, Germany', tags)
    
    # Save to file
    os.makedirs(os.path.dirname(data_file), exist_ok=True)
    gdf_osm.to_file(data_file, driver='GeoJSON')
    print(f"  ✓ Downloaded and saved {len(gdf_osm)} attractions")
else:
    print(f"  ✓ Loading existing data from: {data_file}")
    gdf_osm = gpd.read_file(data_file)

print(f"✓ Loaded {len(gdf_osm)} records")
print(f"✓ Columns: {gdf_osm.shape[1]}")

# ============================================================================
# STEP 2: FIX ID COLUMN (remove element type, keep only numeric ID)
# ============================================================================
print("\n🔧 Step 2: Fixing ID column...")

# Reset index to extract just the numeric ID
gdf_osm = gdf_osm.reset_index(drop=False)

# Extract only the numeric ID (second level of MultiIndex)
if isinstance(gdf_osm.index, pd.MultiIndex):
    gdf_osm['id'] = gdf_osm.index.get_level_values('id')
elif 'id' in gdf_osm.columns:
    # Already has id column
    pass
else:
    # Create sequential ID
    gdf_osm['id'] = range(len(gdf_osm))

# Remove the multi-index, reset to simple index
gdf_osm = gdf_osm.reset_index(drop=True)

print(f"✓ Fixed ID column - now contains only numeric IDs")

# ============================================================================
# STEP 3: DATA TRANSFORMATION
# ============================================================================
print("\n🔄 Step 3: Data Transformation...")

# Create a copy
df_raw = gdf_osm.copy()

# Extract coordinates safely
def extract_coords(geom):
    if geom is None:
        return None, None
    if isinstance(geom, Point):
        return geom.y, geom.x
    else:
        centroid = geom.centroid
        return centroid.y, centroid.x

coords = df_raw['geometry'].apply(extract_coords)
df_raw['lat'] = coords.apply(lambda x: x[0] if x else None)
df_raw['lon'] = coords.apply(lambda x: x[1] if x else None)

print(f"✓ Extracted coordinates for {len(df_raw)} records")

# ============================================================================
# STEP 4: DATA CLEANING
# ============================================================================
print("\n🧹 Step 4: Data Cleaning...")
initial_count = len(df_raw)

# Remove duplicates (but keep geometry in the deduplication)
df_cleaned = df_raw.drop_duplicates(subset=['name', 'lat', 'lon'], keep='first')
print(f"✓ Removed {initial_count - len(df_cleaned)} duplicates")

# ============================================================================
# STEP 5: REVERSE GEOCODING FOR MISSING NAMES
# ============================================================================
print("\n📍 Step 5: Filling missing names with reverse geocoding...")

# Count missing names
missing_names_count = df_cleaned['name'].isna().sum()
print(f"  Found {missing_names_count} records without names")

if missing_names_count > 0:
    # Initialize Nominatim geocoder
    geolocator = Nominatim(user_agent="tourist_attractions_berlin_locator")
    
    def get_name_from_coords(lat, lon):
        """Use reverse geocoding to get a name from coordinates"""
        try:
            location = geolocator.reverse((lat, lon), exactly_one=True, language='de')
            sleep(1)  # Respect API rate limit
            if location and "address" in location.raw:
                address = location.raw["address"]
                # Try to extract a meaningful name
                return (
                    address.get("tourism") or
                    address.get("amenity") or
                    address.get("road") or
                    address.get("suburb") or
                    None
                )
        except Exception as e:
            return None
        return None
    
    # Apply reverse geocoding only to rows with missing names
    # Limit to first 100 to avoid hitting rate limits
    missing_mask = df_cleaned['name'].isna()
    limited_missing = missing_mask & (df_cleaned.index < 100)
    
    print(f"  Attempting reverse geocoding for first 100 missing names...")
    df_cleaned.loc[limited_missing, 'name'] = df_cleaned[limited_missing].apply(
        lambda row: get_name_from_coords(row['lat'], row['lon']) 
        if pd.notnull(row['lat']) else None, 
        axis=1
    )
    
    # Fill remaining with "Unknown"
    df_cleaned['name'] = df_cleaned['name'].fillna('Unknown')
    print(f"✓ Filled missing names")

# ============================================================================
# STEP 6: CATEGORY STANDARDIZATION
# ============================================================================
print("\n🏷️ Step 6: Category Standardization...")

def standardize_category(row):
    if pd.notna(row.get('tourism')):
        return row['tourism']
    elif pd.notna(row.get('historic')):
        return row['historic']
    elif pd.notna(row.get('leisure')):
        return row['leisure']
    else:
        return 'attraction'

df_cleaned['category'] = df_cleaned.apply(standardize_category, axis=1)
print(f"✓ Standardized categories")

# ================================================================================
# STEP 7: DISTRICT AND NEIGHBORHOOD MAPPING
# ================================================================================
print("\n🗺️ Step 7: Mapping districts and neighborhoods...")

# Load Berlin districts GeoDataFrame
districts_file = '../../mapping/lor_ortsteile.geojson'

if os.path.exists(districts_file):
    berlin_districts_gdf = gpd.read_file(districts_file)
    print(f"✓ Loaded Berlin districts data")
    
    # Convert polygon geometries to centroids before spatial join
    # This ensures all geometries are points for accurate district mapping
    df_cleaned['geometry'] = df_cleaned['geometry'].apply(
        lambda geom: geom.centroid if geom.geom_type in ['Polygon', 'MultiPolygon', 'LineString'] else geom
    )
    
    # Perform spatial join
    df_with_districts = gpd.sjoin(
        df_cleaned,
        berlin_districts_gdf[["BEZIRK", "spatial_name", "OTEIL", "geometry"]],
        how="left",
        predicate="within"
    )
    
    # Rename columns
    df_with_districts = df_with_districts.rename(columns={
        "BEZIRK": "district",
        "OTEIL": "neighborhood",
        "spatial_name": "neighborhood_id"
    }).drop(columns=["index_right"], errors='ignore')
    
    # District ID mapping
    district_mapping = {
        'Mitte': '11001001',
        'Friedrichshain-Kreuzberg': '11002002',
        'Pankow': '11003003',
        'Charlottenburg-Wilmersdorf': '11004004',
        'Spandau': '11005005',
        'Steglitz-Zehlendorf': '11006006',
        'Tempelhof-Schöneberg': '11007007',
        'Neukölln': '11008008',
        'Treptow-Köpenick': '11009009',
        'Marzahn-Hellersdorf': '11010010',
        'Lichtenberg': '11011011',
        'Reinickendorf': '11012012'
    }
    
    df_with_districts['district_id'] = df_with_districts['district'].map(district_mapping).astype(str)
    
    print(f"✓ Mapped {df_with_districts['district'].notna().sum()} attractions to districts")
    df_final = df_with_districts
else:
    print(f"⚠️ District file not found: {districts_file}")
    print(f"Continuing without district mapping...")
    df_final = df_cleaned


# ============================================================================
# STEP 8: EXPORT (KEEPING GEOMETRY COLUMN)
# ============================================================================
print("\n💾 Step 8: Exporting...")
output_path = '../../data/processed/tourist_attractions/tourist_attractions_enriched.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Select final columns (INCLUDING GEOMETRY)
final_cols = [
    'id', 'name', 'category', 'lat', 'lon', 'geometry',
    'district', 'district_id', 'neighborhood', 'neighborhood_id',
    'website', 'wheelchair'
]

# Only keep columns that exist
available_cols = [col for col in final_cols if col in df_final.columns]
df_export = df_final[available_cols]

# Convert geometry to WKT string for CSV export
if 'geometry' in df_export.columns:
    df_export['geometry'] = df_export['geometry'].apply(lambda x: x.wkt if x is not None else None)

df_export.to_csv(output_path, index=False)
print(f"✓ Exported {len(df_export)} records to: {output_path}")

print("\n" + "=" * 80)
print("🎉 PIPELINE COMPLETE!")
print("=" * 80)

# Display sample
display(df_export.head())


STARTING DATA PIPELINE

📥 Step 1: Loading/Downloading OSM Data...
  ✓ Loading existing data from: ../../data/raw/tourist_attractions/berlin_attractions_osm.geojson
✓ Loaded 7303 records
✓ Columns: 432

🔧 Step 2: Fixing ID column...
✓ Fixed ID column - now contains only numeric IDs

🔄 Step 3: Data Transformation...
✓ Extracted coordinates for 7303 records

🧹 Step 4: Data Cleaning...
✓ Removed 0 duplicates

📍 Step 5: Filling missing names with reverse geocoding...
  Found 483 records without names
  Attempting reverse geocoding for first 100 missing names...
✓ Filled missing names

🏷️ Step 6: Category Standardization...
✓ Standardized categories

🗺️ Step 7: Mapping districts and neighborhoods...
✓ Loaded Berlin districts data
✓ Mapped 7302 attractions to districts

💾 Step 8: Exporting...
✓ Exported 7303 records to: ../../data/processed/tourist_attractions/tourist_attractions_enriched.csv

🎉 PIPELINE COMPLETE!


,id,name,category,lat,lon,geometry,district,district_id,neighborhood,neighborhood_id,website,wheelchair
0,26972366,Konrad Zuse,memorial,52.522774,13.347221,POINT (13.3472206 52.5227738),Mitte,11001001,Moabit,0102,https://web.archive.org/web/20210131123447/htt...,yes
1,27431777,Luftbrückendenkmal,memorial,52.484135,13.387405,POINT (13.3874049 52.484135),Tempelhof-Schöneberg,11007007,Tempelhof,0703,None,yes
2,29683689,Zentralpunkt Rauenberg,memorial,52.451911,13.366316,POINT (13.3663165 52.4519109),Tempelhof-Schöneberg,11007007,Tempelhof,0703,None,no
3,30815438,Flughafenseesteg,viewpoint,52.569959,13.294084,POINT (13.2940844 52.5699593),Reinickendorf,11012012,Tegel,1202,None,None
4,156850034,Teufelsberg,viewpoint,52.497718,13.242954,POINT (13.2429543 52.4977183),Charlottenburg-Wilmersdorf,11004004,Grunewald,0404,None,limited


: 